# crisp-caption Colab / Kaggle Remote Compute

This notebook runs the remote side of crisp-caption remote compute (works on Colab and Kaggle):

- CrispASR streaming ASR over `wss://.../asr/stream`
- llama.cpp translation over `https://.../v1/chat/completions`
- Cloudflare Tunnel public URL
- Bearer token protection

Run the cells in order. Keep the final cell running while using crisp-caption on Windows.

## 1. GPU Check

In [ ]:
!nvidia-smi

## 2. Clone Helper（免上傳）

Notebook 會自動偵測平台（Colab 或 Kaggle），並 `git clone` 整個 crisp-caption 專案，不需要上傳任何檔案。

In [ ]:
# 免上傳：自動 git clone 整個專案
import os
from pathlib import Path

REPO = "https://github.com/CKwasd/crisp-caption"

# 平台偵測：Colab 用 /content，Kaggle 用 /kaggle/working
if os.path.isdir("/kaggle/working"):
    DEST = Path("/kaggle/working/crisp-caption")
else:
    DEST = Path("/content/crisp-caption")

if not (DEST / "scripts/colab/run_colab_remote.py").exists():
    !git clone --depth 1 {REPO} {str(DEST)}
else:
    print("[skip] crisp-caption already cloned.")

os.chdir(str(DEST))
print("Working dir:", os.getcwd())

## 3. Optional Runtime Sources

The helper automatically downloads model files, cloudflared, CrispASR, and a prebuilt llama.cpp release when possible.

If automatic CrispASR download fails, set one of these before running the service:

- `CRISPASR_EXE=/content/path/to/crispasr`
- `CRISPASR_URL=https://github.com/.../crispasr-linux-...zip`

Choose a llama.cpp backend by setting `LLAMA_BACKEND`:

- `auto`: try ai-dock CUDA, then official Vulkan, then official CPU
- `ai-dock-cuda`: third-party Linux CUDA build from `ai-dock/llama.cpp-cuda`
- `vulkan`: official Ubuntu Vulkan build from `ggml-org/llama.cpp`
- `official-cpu`: official Ubuntu CPU build
- `build-cuda`: build llama.cpp CUDA from source in Colab

If automatic llama.cpp download fails, set one of these:

- `LLAMA_SERVER=/content/path/to/llama-server`
- `LLAMA_CPP_URL=https://github.com/.../llama-...linux...zip`

Leave these empty for automatic latest-release lookup. Use `!python scripts/colab/run_colab_remote.py --build-llama` only if you deliberately want a source build fallback.

In [ ]:
# Examples only. Uncomment and edit if needed.
# %env CRISPASR_EXE=/content/crispasr
# %env CRISPASR_URL=https://github.com/CrispStrobe/CrispASR/releases/download/v0.6.10/crispasr-linux-x86_64-vulkan.zip
%env LLAMA_BACKEND=auto
# %env LLAMA_BACKEND=ai-dock-cuda
# %env LLAMA_BACKEND=vulkan
# %env LLAMA_SERVER=/content/llama-server
# %env LLAMA_CPP_URL=https://github.com/ggml-org/llama.cpp/releases/download/.../llama-...linux...zip

## 4. Start Remote Service

Keep this cell running. It will print the token and Cloudflare URL as large, click-to-select boxes — click each box once, then Ctrl+C to copy. Copy the printed values into Windows:

- `CRISPASR_REMOTE_KEY=...`
- `https://<host>.trycloudflare.com`

In `profiles/profile.ja.json`, set:

```json
"remote_asr_url": "wss://<host>.trycloudflare.com/asr/stream",
"translate_url": "https://<host>.trycloudflare.com/v1/chat/completions"
```

In Windows terminal, set:

```bat
set CRISPASR_REMOTE_KEY=<token>
set OPENAI_API_KEY=<token>
scripts\run-windows.bat
```

In [ ]:
!python scripts/colab/run_colab_remote.py